In [1]:
import os
import pandas as pd
import datasets
from datasets import Dataset, DatasetDict

# 데이터셋 로드 (csv 형식, 구분자는 탭)
raw_datasets = datasets.load_dataset(
    'csv',
    data_files={'train': './data/ratings_train.txt', 'test': './data/ratings_test.txt'},
    delimiter='\t'
)

print("초기 데이터셋 구조:")
print(raw_datasets)
print("\n학습 데이터 샘플:")
print(raw_datasets['train'][0])

# Null 값이나 빈 문자열이 있는 행 제거
# document 컬럼이 유효한(None이 아니고, 길이가 0보다 큰) 샘플만 필터링
def filter_nulls(example):
    return example['document'] is not None and len(example['document']) > 0

cleaned_datasets = raw_datasets.filter(filter_nulls)

print("\n결측치 제거 후 데이터셋 구조:")
print(cleaned_datasets)

초기 데이터셋 구조:
DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})

학습 데이터 샘플:
{'id': 9976970, 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0}

결측치 제거 후 데이터셋 구조:
DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 149995
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 49997
    })
})


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

2025-09-16 14:27:01.786500: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-16 14:27:01.812156: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-16 14:27:01.812195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-16 14:27:01.813395: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-16 14:27:01.819142: I tensorflow/core/platform/cpu_feature_guar

In [3]:
# 토크나이징 함수 정의
def tokenize_function(examples):
    # padding='max_length'는 모든 문장을 지정된 최대 길이(512)로 맞추는 옵션입니다.
    # truncation=True는 최대 길이를 넘어가는 부분을 잘라냅니다.
    return tokenizer(examples['document'], padding="max_length", truncation=True)

# 전체 데이터셋에 토크나이징 함수 적용 (batched=True로 설정하여 병렬 처리로 속도 향상)
tokenized_datasets = cleaned_datasets.map(tokenize_function, batched=True)

# 훈련에 필요 없는 컬럼(id, document) 제거
tokenized_datasets = tokenized_datasets.remove_columns(["id", "document"])
# 훈련을 위해 데이터셋 포맷을 PyTorch 텐서로 변경
tokenized_datasets.set_format("torch")

print("\n데이터셋 전처리(토크나이징) 완료:")
print(tokenized_datasets['train'][0])


데이터셋 전처리(토크나이징) 완료:
{'label': tensor(0), 'input_ids': tensor([   2, 1376,  831, 2604,   18,   18, 4229, 9801, 2075, 2203, 2182, 4243,
           3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    

In [4]:
import numpy as np
import time
from sklearn.metrics import accuracy_score
from transformers import Trainer, TrainingArguments

# 평가 지표(accuracy)를 계산하는 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [5]:
# 1. TrainingArguments 설정 (고정 길이 패딩)
training_args_static = TrainingArguments(
    output_dir="./results_static_padding",  # 모델과 결과물이 저장될 디렉토리
    eval_strategy="epoch",            # 매 에폭마다 평가 수행
    save_strategy="epoch",                  # 매 에폭마다 모델 저장
    per_device_train_batch_size=64,         # 훈련용 배치 사이즈
    per_device_eval_batch_size=32,          # 평가용 배치 사이즈
    num_train_epochs=1,                     # 총 훈련 에폭 수 (1 에폭으로도 충분히 좋은 성능이 나옴)
    learning_rate=2e-5,                     # 학습률
    weight_decay=0.01,                      # 가중치 감소
    load_best_model_at_end=True,            # 훈련 종료 후 가장 좋은 모델을 로드
    metric_for_best_model="accuracy",       # 최고 모델 선정 기준
    bf16=True,
)

# 2. Trainer 초기화
trainer_static = Trainer(
    model=model,
    args=training_args_static,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

In [7]:
# 3. 모델 학습 시작
print("\n--- STEP 4: 고정 패딩(Static Padding)으로 모델 학습 시작 ---")
trainer_static.train()


--- STEP 4: 고정 패딩(Static Padding)으로 모델 학습 시작 ---


Epoch,Training Loss,Validation Loss,Accuracy
1,0.220400,0.234296,0.903934


TrainOutput(global_step=2344, training_loss=0.21364967163919182, metrics={'train_runtime': 1385.0711, 'train_samples_per_second': 108.294, 'train_steps_per_second': 1.692, 'total_flos': 3.94653427487232e+16, 'train_loss': 0.21364967163919182, 'epoch': 1.0})

- epoch당 23분 5초 소요

In [8]:
# 4. 모델 평가
print("\n--- STEP 4: 학습 완료 및 평가 ---")
eval_results_static = trainer_static.evaluate()
print(f"평가 결과 (정확도): {eval_results_static['eval_accuracy']:.4f}")


--- STEP 4: 학습 완료 및 평가 ---


평가 결과 (정확도): 0.9039


- 2epoch만에 정확도 0.9039 달성
- epoch을 늘리면 정확도가 더 증가할 여지가 있으나 2epoch까지만 학습

- Bucketing 적용 및 결과 비교

In [20]:
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [21]:
from transformers import DataCollatorWithPadding

# 공정한 비교를 위해 모델을 다시 초기 상태로 불러옵니다.
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Dynamic Padding을 위한 Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 1. TrainingArguments 설정 (Bucketing 적용)
training_args_bucket = TrainingArguments(
    output_dir="./results_bucketing",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    group_by_length=True,  # Bucketing 옵션 활성화
    bf16=True,
)

# 2. Trainer 초기화 (Data Collator 추가)
trainer_bucket = Trainer(
    model=model,
    args=training_args_bucket,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer, # Data Collator가 tokenizer 정보를 필요로 함
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_81244/2969059163.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_bucket = Trainer(


In [22]:
# 3. 모델 학습 시작
print("\n--- STEP 5: Bucketing과 Dynamic Padding으로 모델 학습 시작 ---")
trainer_bucket.train()


--- STEP 5: Bucketing과 Dynamic Padding으로 모델 학습 시작 ---


Epoch,Training Loss,Validation Loss,Accuracy
1,0.251600,0.241233,0.902934
2,0.177000,0.239602,0.906114


TrainOutput(global_step=4688, training_loss=0.22956441286887733, metrics={'train_runtime': 2829.8156, 'train_samples_per_second': 106.01, 'train_steps_per_second': 1.657, 'total_flos': 7.89306854974464e+16, 'train_loss': 0.22956441286887733, 'epoch': 2.0})

- epoch당 23분 44초 소요

In [23]:
# 4. 모델 평가
print("\n--- STEP 5: 학습 완료 및 평가 ---")
eval_results_bucket = trainer_bucket.evaluate()
print(f"평가 결과 (정확도): {eval_results_bucket['eval_accuracy']:.4f}")


--- STEP 5: 학습 완료 및 평가 ---


평가 결과 (정확도): 0.9061


- 2epoch만에 정확도 0.9061 달성

- 전체 학습시간은 Bucketing을 적용한 경우가 조금이지만 더 오래걸렸다.
- 일반적으로 불필요한 패딩 연산을 제거하므로 훈련 시간을 단축시키는 것으로 알려져있다.
- train_steps_per_second를 보면, 적용전: 1.692, 적용 후: 1.657로 오히려 2%가량 더 느려졌다.
- 원인은 크게 2가지로 생각해 볼 수 있다.

1. BFloat16 타입을 사용해 이미 효율적으로 연산을 하기때문에 효과가 적게 나타났다.
2. NSMC 데이터셋의 영화 리뷰들의 대부분의 길이가 비슷하다. 따라서 패딩 감소로 얻는 이점보다 길이별로 그룹화하는데 드는 비용이 더 컸을 것이다.

- 성능 또한 bucketing을 적용한 쪽이 미세하게 더 좋다.
- 성능은 Bucketing 여부와 크게 관련이 없는 것으로 알려져있는데 0.3%정도의 차이로 무시할 수 있는 수준이다.
- 하지만 Training Loss, Validation Loss를 보면, 적용한 후가 격차가 크게 벌어진 것을 볼 수 있다.
- epoch을 더 늘렸을 때를 고려해보면, Bucketing을 적용하지 않은 것이 더 좋을 가능성이 높을 것 같다.